# 02 – Datenbereinigung

Die eigentliche Bereinigung liegt in `src/spotify_analysis/data.py` (auch als Skript: `python scripts/clean_data.py`). Dieses Notebook zeigt die Probleme der Rohdaten und schreibt `data/processed/spotify-2023-cleaned.csv`.

In [1]:
import sys

sys.path.insert(0, "../src")

import pandas as pd
from spotify_analysis.data import PROCESSED_PATH, clean, load_raw

df_raw = load_raw()
df_raw.shape

(953, 24)

## Probleme der Rohdaten

In [2]:
streams = pd.to_numeric(df_raw["streams"], errors="coerce")
print("Streams nicht numerisch:", streams.isna().sum())
df_raw.loc[streams.isna(), ["track_name", "artist(s)_name", "streams"]]

Streams nicht numerisch: 1


,track_name,artist(s)_name,streams
574,Love Grows (Where My Rosemary Goes),Edison Lighthouse,BPM110KeyAModeMajorDanceability53Valence75Ener...


In [3]:
print("Fehlende Keys:", df_raw["key"].isna().sum())
print("Fehlende in_shazam_charts:", df_raw["in_shazam_charts"].isna().sum())
print("Duplikate (Titel + Künstler):", df_raw.duplicated(["track_name", "artist(s)_name"]).sum())
df_raw[df_raw.duplicated(["track_name", "artist(s)_name"], keep=False)].sort_values("track_name")[["track_name", "artist(s)_name", "streams"]]

Fehlende Keys: 95
Fehlende in_shazam_charts: 50
Duplikate (Titel + Künstler): 4


,track_name,artist(s)_name,streams
372,About Damn Time,Lizzo,723894473
764,About Damn Time,Lizzo,723894473
178,SNAP,Rosa Linn,726307468
873,SNAP,Rosa Linn,711366595
345,SPIT IN MY FACE!,ThxSoMch,303216294
482,SPIT IN MY FACE!,ThxSoMch,301869854
512,Take My Breath,The Weeknd,130655803
616,Take My Breath,The Weeknd,432702334


In [4]:
broken = df_raw[["track_name", "artist(s)_name"]].apply(lambda s: s.str.contains("\ufffd")).any(axis=1)
print("Zeilen mit beschädigten Zeichen (U+FFFD):", broken.sum())
df_raw.loc[broken, ["track_name", "artist(s)_name"]].head()

Zeilen mit beschädigten Zeichen (U+FFFD): 109


,track_name,artist(s)_name
21,I Can See You (Taylor���s Version) (From The,Taylor Swift
26,Calm Down (with Selena Gomez),"R��ma, Selena G"
36,Fr��gil (feat. Grupo Front,"Yahritza Y Su Esencia, Grupo Frontera"
60,T��,"dennis, MC Kevin o Chris"
63,BESO,"Rauw Alejandro, ROSAL�"


Die Zeichen sind bereits im Kaggle-Export verloren gegangen (`Taylor’s Version` → `Taylor��s Version`) und lassen sich nicht zuverlässig rekonstruieren. Sie werden zu einem einzelnen `�` zusammengefasst.

## Bereinigung

- `streams` und Spalten mit Tausendertrennzeichen → numerisch (die eine kaputte Zeile wird `NaN`)
- fehlende `key`-Werte → `Unknown` (statt pauschal `C`, was die Key-Statistik verfälschen würde)
- doppelte Titel/Künstler-Paare → die Zeile mit den meisten Streams bleibt

In [5]:
df = clean(df_raw)
print(df_raw.shape, "->", df.shape)
df.dtypes

(953, 24) -> (949, 24)


track_name                str
artist(s)_name            str
artist_count            int64
released_year           int64
released_month          int64
released_day            int64
in_spotify_playlists    int64
in_spotify_charts       int64
streams                 Int64
in_apple_playlists      int64
in_apple_charts         int64
in_deezer_playlists     Int64
in_deezer_charts        int64
in_shazam_charts        Int64
bpm                     int64
key                       str
mode                      str
danceability_%          int64
valence_%               int64
energy_%                int64
acousticness_%          int64
instrumentalness_%      int64
liveness_%              int64
speechiness_%           int64
dtype: object

In [6]:
df["key"].value_counts()

key
C#         119
G           96
Unknown     94
G#          91
F           89
B           81
D           81
A           75
F#          73
E           62
A#          55
D#          33
Name: count, dtype: int64

In [7]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False, encoding="utf-8", lineterminator="\n")
PROCESSED_PATH

WindowsPath('C:/Spotify/data/processed/spotify-2023-cleaned.csv')